In [ ]:
import os
from dotenv import load_dotenv
import requests

load_dotenv()  # loads .env file in current directory

In [ ]:
# Basic configuration for Todoist API
API_TOKEN = os.getenv("TODOIST_API_TOKEN")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

BASE_URL = "https://api.todoist.com/api/v1"

# Finding Projects

In [ ]:
# Project Functions

def get_projects():
    url = f"{BASE_URL}/projects"
    all_projects = []
    cursor = None

    while True:
        params = {"limit": 50}
        if cursor:
            params["cursor"] = cursor

        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()

        all_projects.extend(data["results"])

        cursor = data.get("next_cursor")
        if not cursor:
            break

    return all_projects

In [ ]:
projects = get_projects()
project_summary = {p["name"]: p["id"] for p in projects}

project_WR = project_summary.get("Weekly review")
assert project_WR is not None, "Project 'Weekly review' not found"

# Adding Sections

In [ ]:
# Section Functions

def get_sections():
    url = f"{BASE_URL}/sections"
    all_sections = []
    cursor = None

    while True:
        params = {"limit": 50}
        if cursor:
            params["cursor"] = cursor

        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()

        all_sections.extend(data["results"])

        cursor = data.get("next_cursor")
        if not cursor:
            break

    return all_sections

def create_section(section):
    url = f"{BASE_URL}/sections"
    response = requests.post(url, headers=HEADERS, json=section)
    response.raise_for_status()
    return response.json()

def section_exists(section, existing_sections):
    return any(
        (s["name"] == section["name"]) and
        (s["project_id"] == section["project_id"])
        for s in existing_sections
    )

In [ ]:
sections_to_add = [
    {"name": "Monthly Review", "project_id": project_WR},
]

In [ ]:
existing_sections = get_sections()
existing_sections = [s for s in existing_sections if s["project_id"] == project_WR]

for section in sections_to_add:
    if not section_exists(section, existing_sections):
        response = create_section(section)
        print(response)
    else:
        print(f"Section already exists: {section['name']}")

In [ ]:
existing_sections = get_sections()
sections_WR = [s for s in existing_sections if s["project_id"] == project_WR]
section_summary = {s["name"]: s["id"] for s in sections_WR}

sec_monthly_review = section_summary.get("Monthly Review")
assert sec_monthly_review is not None, "Missing section: Monthly Review"

# Adding Tasks

In [ ]:
# Task Functions

def task_exists(task, existing_tasks):
    return any(
        (t["content"] == task["content"]) and
        (t.get("project_id") == task.get("project_id"))
        for t in existing_tasks
    )

def get_tasks(project_id):
    url = f"{BASE_URL}/tasks"
    response = requests.get(url, headers=HEADERS, params={"project_id": project_id})
    response.raise_for_status()
    return response.json()["results"]

def create_task(task):
    url = f"{BASE_URL}/tasks"
    response = requests.post(url, headers=HEADERS, json=task)
    response.raise_for_status()
    return response.json()

In [ ]:
tasks_to_add = [
    {
        "content": "Review Someday/Maybe",
        "project_id": project_WR,
        "section_id": sec_monthly_review
    },
    {
        "content": "Review Deferred Academic Projects",
        "project_id": project_WR,
        "section_id": sec_monthly_review    
    }, 
    {   
        "content": "Review Deferred Projects",
        "project_id": project_WR,
        "section_id": sec_monthly_review    
    },
    {
        "content": "Review All Labels and Delete Unused Ones",
        "project_id": project_WR,
        "section_id": sec_monthly_review
    }
]

In [ ]:
existing_tasks = get_tasks(project_WR)

In [ ]:
for task in tasks_to_add:
    if not task_exists(task, existing_tasks):
        response = create_task(task)
        print(f"Created: {task['content']}")
    else:
        print(f"Already exists: {task['content']}")